In [1]:
import struct
import numpy as np
import math
from enum import Enum
import sys
sys.path.append('/Users/abhinavarora/Desktop/Machine Learning')
import custom_math
import random

#Parsing the binary format
with open("/Users/abhinavarora/Desktop/Machine Learning/Neural Network/MNIST handwritten /train-images-idx3-ubyte/train-images-idx3-ubyte", "rb") as f:
    magic, num, rows, cols = struct.unpack(">IIII", f.read(16))
    images = np.frombuffer(f.read(), dtype=np.uint8)
    images = images.reshape(num, rows * cols)

#Loading in the labels
with open("/Users/abhinavarora/Desktop/Machine Learning/Neural Network/MNIST handwritten /train-labels-idx1-ubyte/train-labels-idx1-ubyte", "rb") as f:
    magic, num = struct.unpack(">II", f.read(8))
    labels = np.frombuffer(f.read(), dtype=np.uint8)


#This is to one-hot encode input labels whenever softmax regression is used.
#The class number simply is the number of classes
def vectorize_labels(input_labels, class_number):
    #The one-hot encoded arrays are being stored in a matrix where each column corresponds to the number of classes
    #And the row corresponds to which class-type. This is consistent with the dimensions of the output of the final layer
    res = [[0] * len(input_labels) for _ in range(class_number)]
    for i in range(len(input_labels)):
        #one-hot encoding
        res[input_labels[i]][i] = 1
    
    return res

vectorized_labels = vectorize_labels(labels, 10)

In [ ]:
#Dimension of the vector
img_dimension = rows * cols
#Each image flattened into a vector and put into a matrix
image_input_matrix = [[0] * len(images) for _ in range(img_dimension)]

for column in range(len(images)):
    for row in range(len(images[0])):
        #Normalizing to fit every value between 0 and 1 to solve vanishing gradients
        image_input_matrix[row][column] = images[row][column]/255

In [ ]:
class ActivationFunctions:
    #Required to know the number of neurons since activation functions will be vector operations 
    def __init__(self, input_matrix):
        self.input_matrix = input_matrix

    def sigmoid(self, input_val):
        exponent = math.exp(-input_val)
        return (1/(1+exponent))
    
    def ReLU(self, input_val):
        if input_val > 0:
            return input_val
        return 0

    
class FunctionType(Enum):
    RELU = 1
    SIGMOID = 2

#This is a layer of neurons with specific number of neurons that is a hyper parameter
#As well as the number of inputs this layer will take
class Layer:
    def __init__(self, neuron_num, input_size, function_type: FunctionType):
        self.function_type = function_type
        self.neuron_num = neuron_num
        self.input_size = input_size
        #Generates a random number in the normal distribution with mean 0 and std 0.01. This results in values between
        #-0.03 and 0.03. This is required since we want to break symmetry in each layer of a neural network and not 
        #make them identical
        self.parameters = [[random.Random(float).gauss(0, 0.01)] * input_size for _ in range(neuron_num)]
        self.bias = [[0] * 1 for _ in range(neuron_num)]
        #Linear part and result of the activation being stored directly as a property of the layer itself
        self.z = None
        self.a = None
    
    def execute_function(self, input_val):
        #Initialising the class of the different activation functions
        functions = ActivationFunctions()
        if self.function_type == FunctionType.RELU:
            return functions.ReLU(input_val)
        if self.function_type == FunctionType.SIGMOID:
            return functions.sigmoid(input_val)
    
    #Returns the omega * x + b
    #Bias automatically gets broadcasted in this function
    def linearize(self, parameters, input, bias):
        parameter_input_product = custom_math.matrix_with_matrix_multiplication(parameters, input)
        #Broadcast
        #This is the number of rows of the broadcasted bias matrix
        dimension = len(parameter_input_product[0])
        broadcasted_matrix = [[0] * len(dimension) for _ in range(parameter_input_product)]
        for row in range(len(broadcasted_matrix)):
            for col in range(len(broadcasted_matrix[0])):
                broadcasted_matrix[row][col] = bias[row]
        return custom_math.matrix_addition_and_sub(parameter_input_product, broadcasted_matrix, "add")

    #The softmax function is only for the last layer's output, by default it will be set to 0. However, the underlying
    #hypothesis function will not change
    def hypothesis(self, linear, softmax=False):
        linear_copy = [[0] * len(linear[0]) for _ in range(len(linear))]
        for row in range(len(linear)):
            for col in range(len(linear[0])):
                linear_copy[row][col] = self.execute_function(linear[row][col])
        
        if not softmax:
            return linear_copy
        else:
            for col in range(len(linear[0])):
                #Calculating the total exponential sum for each ouptut
                exponential_sum = 0
                for row in range(len(linear)):
                    exponential_sum += math.exp(linear_copy[row][col])
                
                #Applying the softmax formula
                for row in range(len(linear)):
                    linear_copy[row][col] = math.exp(linear_copy[row][col])/exponential_sum
        
        return linear_copy
    
class LossFunctions:
    def __init__(self):
        pass

    #This is the cross entropy function required
    #y_hat is the final prediction vector and y is the actual label vector
    def cross_entropy_loss(self, y_hat, y):
        regresularised_predictions = self.regularise(y_hat)
        for i in range(len(regresularised_predictions)):
            regresularised_predictions[i] = math.log(regresularised_predictions[i]) * y[i]
        
        res = sum(i for i in regresularised_predictions)
        return -res
    
    #This is a function that will add an epsilon value (extremely small) to prevent from obtaining the log (0) in any calculation
    def regularise(vector, epsilon = 1e-5):
        vector_copy = [0] * len(vector)
        for i in range(len(vector)):
            vector_copy[i] = vector[i] + epsilon
        
        return vector_copy
        
        

In [ ]:
class Network:
    def __init__(self, layer_num, neurons_in_layers, initial_input):
        #This initialises the number of layers
        self.number_of_layers = layer_num
        #This is a list that specifies the number of neurons in each layer 
        self.neurons_in_layers = neurons_in_layers
        self.inital_input = initial_input
        #This is the array that stores the actual layer objects
        self.layers = []
        #Initialising the layers
        for i in range(len(self.number_of_layers)):
            #This is specifically for the first layer. This is because the input_size is the dimension of the vector of each training example
            if i == 0:
                layer = Layer(self.neurons_in_layers[i], len(self.inital_input), FunctionType.SIGMOID)
            #For the other layers, the input size the number of neurons of the previous layer since each neuron outputs a single number
            else:
                layer = Layer(self.neurons_in_layers[i], self.neurons_in_layers[i-1], FunctionType.SIGMOID)
            
            self.layers.append(layer)

    #This is the feedforward function. This will be a recursive function.
    #The layer_index specifies which layer in self.layers and input specifies the input for each layer
    def feedforward(self, layer_index, input):
        #Base case
        if layer_index >= len(self.layers):
            #Technically this is the final output now
            return input
        layer:Layer = self.layers[layer_index]
        linear_res = layer.linearize(layer.parameters, input, layer.bias)
        output = layer.hypothesis(linear_res)
        #Useful for caching results
        layer.a = output
        layer.z = linear_res
        #The input of the next layer becomes the output of the current layer
        return self.feedforward(layer_index + 1, output)
    
    #THe total loss function is the sum of the loss functions across each layer.
    #The output is the output of the final layer
    def total_loss(self, output, loss_type, input_labels):
        total_loss = 0
        #The output has dimension (number of neurons in last layer, training examples) but this makes it hard to iterate over each column
        #Therefore, the transpose allows us to iterate row by row
        #Same logic for input_labels since they were one-hot encoded to be in the same dimension as the output
        output_transpose = custom_math.transpose_matrix(output)
        input_labels_transpose = custom_math.transpose_matrix(input_labels)
        for row in range(len(output_transpose)):
            total_loss += loss_type(output_transpose[row], input_labels_transpose[row])
        
        total_loss /= len(input_labels_transpose)
        return total_loss
    
    #The following backprop functions are hardcoded for binary cross entropy. It is not practical to have such hardcoded backprop functions 
    #and so an autograd engine will be implemented later on
    def last_layer_backprop(self, labels, final_layer:Layer, prev_layer: Layer):
        prev_activation_transpose = custom_math.transpose_matrix(prev_layer.a)
        #First sum calculates the loss w.r.t the first training example. This is also to store the iterative results of the upcoming addition calculations
        #for the total loss in final_first_sum
        first_sum = custom_math.matrix_with_matrix_multiplication(prev_activation_transpose, custom_math.calculate_vector(labels[0], final_layer.a))
        final_first_sum = custom_math.scalar_multiply_matrix(first_sum, -1)
        for i in range(1, len(labels)):
            next_matrix = custom_math.matrix_with_matrix_multiplication(prev_activation_transpose, custom_math.calculate_vector(labels[i], final_layer.a))
            final_next_matrix = custom_math.scalar_multiply_matrix(next_matrix, -1)
            #Accumulation of sum as described above
            final_first_sum = custom_math.matrix_addition_and_sub(final_first_sum, final_next_matrix, "add")
        
        #The total loss is the average of losses across each training example
        res = custom_math.scalar_multiply_matrix(first_sum, -1/len(labels))
        return res
    
    #This is the general pattern for the backprop for previous layers. 
    def previous_layer_backprop(self, current_layer:Layer, next_layer:Layer, previous_product,  previous_layer:Layer = None):
        next_layer_bias_transpose = custom_math.transpose_matrix(next_layer.bias)
        matrix_of_ones = [[1] * len(current_layer.a[0]) for _ in range(len(current_layer.a))]
        one_minus_a = custom_math.matrix_addition_and_sub(matrix_of_ones, current_layer.a)
        product_one = custom_math.matrix_with_matrix_multiplication(current_layer.a, one_minus_a)
        product_two = custom_math.matrix_with_matrix_multiplication(next_layer_bias_transpose, product_one)
        #This check is necessary since the previous_layer's activation output is the current layer's input
        #However, there won't be a previous layer for the first layer so that uses the initial input
        if previous_layer != None:
            product_three = custom_math.matrix_with_matrix_multiplication(product_two, previous_layer.a)
        else:
            product_three = custom_math.matrix_with_matrix_multiplication(product_two, self.inital_input)
        final_product = custom_math.matrix_with_matrix_multiplication(previous_product, product_three)
        return final_product
    

